# Notebook 3b — Per-File Analysis (Tables Only)

**Purpose:** Paper-ready **tabular** version of `per_file_analysis.ipynb` — same information, no charts. All numeric outputs are rounded to two decimals on a 0–100 scale (F-measure / Precision / Recall) or shown as integer counts (TP / FP / FN).

**Input:** Same `stdout.log` files as `per_file_analysis.ipynb`.

**Definitions:**
- **Per-file F-measure:** F = TP / (TP + 0.5·(FP + FN)), from that file's own event counts. Equivalently the harmonic mean of the file's own precision and recall — no averaging across files is involved.
- **Mean (arith.) column of Table 4:** unweighted arithmetic mean of the per-subset F-measures. The official DCASE score takes a *harmonic* mean of those same per-subset values, so this column is not the reported metric.

**Outputs (all tabular):**
1. Per-file F-measure (files × experiments)
2. Per-file Recall (files × experiments)
3. Per-file Precision (files × experiments)
4. Species / subset breakdown — F, P, R aggregated per subset
5. Top-20 files by mean FP across experiments
6. TP / FP / FN for a selected file across experiments
7. Δ F-measure vs Baseline (per file × experiment)
8. LaTeX export of the subset-level and per-file F-measure tables

## 0 — Configuration

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))
import parse_logs as pl

# ── Experiment log paths ─────────────────────────────────────────
EXPERIMENT_LOGS = {
    "Baseline":            "/media/helios/Masters/Repository/dcase-few-shot-bioacoustic/baselines/dcase2024_task5/logs/experiments/runs/baseline_seed1234/2026-09-17_09-08-46/stdout.log",  # e.g. "../logs/experiments/runs/baseline/2026-.../stdout.log"
    "CIFE":                "/media/helios/Masters/Repository/dcase-few-shot-bioacoustic/baselines/dcase2024_task5/logs/experiments/runs/cife_seed1234/2026-09-17_22-11-22/stdout.log",
    "Slot Uniform":        "/media/helios/Masters/Repository/dcase-few-shot-bioacoustic/baselines/dcase2024_task5/logs/experiments/runs/slot_uniform_seed1234/2026-09-18_12-05-18/stdout.log",
    "Phase 1 (Random)":    "/media/helios/Masters/Repository/dcase-few-shot-bioacoustic/baselines/dcase2024_task5/logs/experiments/runs/phase1_random_seed1234/2026-09-18_05-15-56/stdout.log",
    "Phase 2 (Energy)":    "/media/helios/Masters/Repository/dcase-few-shot-bioacoustic/baselines/dcase2024_task5/logs/experiments/runs/phase2_energy_seed1234/2026-09-18_16-56-51/stdout.log",
    "Phase 3 (Activation)": "/media/helios/Masters/Repository/dcase-few-shot-bioacoustic/baselines/dcase2024_task5/logs/experiments/runs/phase3_seed1234_output.log",
    "AFRI (Direct)": "/media/helios/Masters/Repository/dcase-few-shot-bioacoustic/baselines/dcase2024_task5/logs/experiments/runs/afri_direct_seed1234/2026-09-19_14-20-53/stdout.log",
    "AFRI (Inverse)": "/media/helios/Masters/Repository/dcase-few-shot-bioacoustic/baselines/dcase2024_task5/logs/experiments/runs/afri_inverse_seed1234/2026-09-19_20-45-13/stdout.log",
}


# Rounding used throughout the notebook (typical for a research paper).
N_DECIMALS = 2

# Change if your baseline is labeled differently.
BASELINE_LABEL = "Baseline"

logs_to_parse = {k: v for k, v in EXPERIMENT_LOGS.items() if v is not None}
print(f"Parsing {len(logs_to_parse)} experiment(s)...")

# ── Paper tables output directory ──────────────────────────
# Every table produced below is also written as a stand-alone .tex file
# (\begin{table} block) under paper/tables/.
PAPER_TABLES_DIR = Path("../../../paper/tables").resolve()
PAPER_TABLES_DIR.mkdir(parents=True, exist_ok=True)

def save_tex(df, filename, caption, label, **to_latex_kwargs):
    """Write `df` as a standalone LaTeX table to PAPER_TABLES_DIR/filename."""
    kwargs = dict(
        float_format=f"%.{N_DECIMALS}f",
        na_rep="—",
        caption=caption,
        label=label,
    )
    kwargs.update(to_latex_kwargs)
    out = PAPER_TABLES_DIR / filename
    out.write_text(df.to_latex(**kwargs))
    print(f"  ✓ Saved {out.name}")


Parsing 8 experiment(s)...


## 1 — Load & Parse Logs

In [2]:
import pandas as pd
import numpy as np

experiments = pl.load_experiments(logs_to_parse)

exp_labels = [k for k in EXPERIMENT_LOGS.keys() if experiments.get(k) is not None]

df_pf = pl.per_file_df(experiments)

# Rescale precision/recall/fmeasure from 0–1 to 0–100 for paper-friendly display.
for col in ["precision", "recall", "fmeasure"]:
    if col in df_pf.columns and not df_pf.empty:
        col_max = df_pf[col].dropna().max()
        if pd.notna(col_max) and col_max <= 1.01:
            df_pf[col] = df_pf[col] * 100

print(f"Per-file rows: {len(df_pf)}")
if not df_pf.empty:
    print(f"Files found  : {df_pf['filename'].nunique()}")
    print(f"Subsets      : {sorted(df_pf['subset'].unique())}")
df_pf.head()

  ✓ Baseline                        fmeasure=16.615
  ✓ CIFE                            fmeasure=15.296
  ✓ Slot Uniform                    fmeasure=15.392
  ✓ Phase 1 (Random)                fmeasure=15.618
  ✓ Phase 2 (Energy)                fmeasure=15.434
  ✓ Phase 3 (Activation)            fmeasure=15.304
  ✓ AFRI (Direct)                   fmeasure=15.502
  ✓ AFRI (Inverse)                  fmeasure=15.613
Per-file rows: 336
Files found  : 42
Subsets      : ['HB', 'ME', 'PB', 'PB24', 'PW', 'RD']


,from_backfill,experiment,filename,subset,TP,FP,FN,total_n_pos_events,precision,recall,fmeasure
0,False,Baseline,BUK1_20181011_001004.wav,PB,3,31,28,31,8.823529,9.677419,9.230769
1,False,Baseline,BUK4_20161011_000804.wav,PB,0,164,47,47,0.000000,0.000000,0.000000
2,False,Baseline,BUK4_20161031_232104a.wav,PB24,22,20,155,177,52.380952,12.429379,20.091324
3,False,Baseline,BUK4_20171022_004304a.wav,PB,5,7,12,17,41.666667,29.411765,34.482759
4,False,Baseline,BUK5_20161101_002104a.wav,PB,19,152,69,88,11.111111,21.590909,14.671815


## 2 — Per-File F-measure (files × experiments)

Tabular replacement for the grouped bar chart and the F-measure heatmap. Rows are sorted by mean F-measure (ascending) so the worst files sit at the top — matching the heatmap ordering.

In [3]:
if df_pf.empty:
    print("Per-file data not available.")
else:
    pivot_fm = (
        df_pf.pivot_table(index="filename", columns="experiment", values="fmeasure")
        .reindex(columns=exp_labels)
    )
    pivot_fm = pivot_fm.loc[pivot_fm.mean(axis=1).sort_values().index]
    pivot_fm["Mean"] = pivot_fm.mean(axis=1)
    pivot_fm = pivot_fm.round(N_DECIMALS)

    print("=== Table 1 — Per-File F-measure (0–100, sorted worst → best by mean) ===")
    display(pivot_fm)

    save_tex(
        pivot_fm,
        "per_file_fmeasure.tex",
        caption=(
            "Per-file F-measure for every experiment on the DCASE 2024 Task~5 "
            "validation set. F-measure computed as "
            "$\\mathrm{TP} / (\\mathrm{TP} + 0.5(\\mathrm{FP} + \\mathrm{FN}))$; "
            "rows sorted ascending by row mean. All values on a 0--100 scale."
        ),
        label="tab:per_file_fmeasure",
    )

=== Table 1 — Per-File F-measure (0–100, sorted worst → best by mean) ===


experiment,Baseline,CIFE,Slot Uniform,Phase 1 (Random),Phase 2 (Energy),Phase 3 (Activation),AFRI (Direct),AFRI (Inverse),Mean
filename,,,,,,,,,
BUK4_20161011_000804.wav,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
BUK4_20160925_231604.wav,0.00,0.00,0.00,0.00,4.76,0.00,0.00,0.00,0.60
pw4.wav,2.21,2.15,2.18,2.04,2.18,2.19,2.19,2.05,2.15
pw5.wav,3.54,3.81,3.79,4.14,3.72,3.79,3.82,4.62,3.90
pw2.wav,4.30,4.40,4.31,4.92,4.33,4.45,5.54,4.82,4.63
ME1.wav,5.56,18.95,4.23,4.24,4.28,3.75,5.06,3.55,6.20
pw6.wav,6.92,5.58,5.93,6.44,5.56,7.54,6.81,7.28,6.51
pw1.wav,6.36,7.37,7.36,7.41,7.28,7.40,7.23,7.37,7.22
pw15.wav,8.30,7.50,8.42,8.49,8.50,8.43,8.18,8.32,8.27


  ✓ Saved per_file_fmeasure.tex


## 3 — Per-File Recall (files × experiments)

Tabular replacement for the recall heatmap. Rows sorted by mean recall (ascending).

In [4]:
if not df_pf.empty:
    pivot_recall = (
        df_pf.pivot_table(index="filename", columns="experiment", values="recall")
        .reindex(columns=exp_labels)
    )
    pivot_recall = pivot_recall.loc[pivot_recall.mean(axis=1).sort_values().index]
    pivot_recall["Mean"] = pivot_recall.mean(axis=1)
    pivot_recall = pivot_recall.round(N_DECIMALS)

    print("=== Table 2 — Per-File Recall (0–100, sorted worst → best by mean) ===")
    display(pivot_recall)

    save_tex(
        pivot_recall,
        "per_file_recall.tex",
        caption=(
            "Per-file recall for every experiment on the DCASE 2024 Task~5 "
            "validation set. Rows sorted ascending by row mean. "
            "All values on a 0--100 scale."
        ),
        label="tab:per_file_recall",
    )

=== Table 2 — Per-File Recall (0–100, sorted worst → best by mean) ===


experiment,Baseline,CIFE,Slot Uniform,Phase 1 (Random),Phase 2 (Energy),Phase 3 (Activation),AFRI (Direct),AFRI (Inverse),Mean
filename,,,,,,,,,
BUK4_20161011_000804.wav,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
BUK4_20160925_231604.wav,0.00,0.00,0.00,0.00,2.70,0.00,0.00,0.00,0.34
BUK1_20181013_023504.wav,0.00,0.00,12.50,12.50,12.50,0.00,0.00,8.33,5.73
BUK5_20181007_011905.wav,9.09,7.58,7.58,7.58,7.58,7.58,9.09,7.58,7.95
file_97_113.wav,8.33,10.00,9.17,9.17,9.17,9.17,8.33,9.17,9.06
BUK4_20161031_232104a.wav,12.43,6.21,12.43,12.99,12.99,9.04,11.86,12.99,11.37
BUK1_20181011_001004.wav,9.68,6.45,16.13,12.90,16.13,16.13,19.35,9.68,13.31
BUK5_20180921_015906a.wav,17.39,13.04,17.39,17.39,17.39,17.39,17.39,17.39,16.85
BUK5_20161101_002104a.wav,21.59,19.32,21.59,21.59,21.59,15.91,17.05,25.00,20.45


  ✓ Saved per_file_recall.tex


## 4 — Per-File Precision (files × experiments)

In [5]:
if not df_pf.empty:
    pivot_prec = (
        df_pf.pivot_table(index="filename", columns="experiment", values="precision")
        .reindex(columns=exp_labels)
    )
    pivot_prec = pivot_prec.loc[pivot_prec.mean(axis=1).sort_values().index]
    pivot_prec["Mean"] = pivot_prec.mean(axis=1)
    pivot_prec = pivot_prec.round(N_DECIMALS)

    print("=== Table 3 — Per-File Precision (0–100, sorted worst → best by mean) ===")
    display(pivot_prec)

    save_tex(
        pivot_prec,
        "per_file_precision.tex",
        caption=(
            "Per-file precision for every experiment on the DCASE 2024 Task~5 "
            "validation set. Rows sorted ascending by row mean. "
            "All values on a 0--100 scale."
        ),
        label="tab:per_file_precision",
    )

=== Table 3 — Per-File Precision (0–100, sorted worst → best by mean) ===


experiment,Baseline,CIFE,Slot Uniform,Phase 1 (Random),Phase 2 (Energy),Phase 3 (Activation),AFRI (Direct),AFRI (Inverse),Mean
filename,,,,,,,,,
BUK4_20161011_000804.wav,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
pw4.wav,1.14,1.11,1.13,1.05,1.12,1.13,1.13,1.06,1.11
pw5.wav,1.81,1.94,1.94,2.12,1.90,1.94,1.95,2.37,2.00
pw2.wav,2.23,2.27,2.24,2.58,2.24,2.32,2.89,2.52,2.41
BUK4_20160925_231604.wav,0.00,0.00,0.00,0.00,20.00,0.00,0.00,0.00,2.50
ME1.wav,2.93,10.71,2.19,2.19,2.22,1.95,2.65,1.83,3.34
pw6.wav,3.59,2.88,3.06,3.33,2.86,3.92,3.53,3.78,3.37
pw1.wav,3.31,3.84,3.83,3.86,3.80,3.86,3.77,3.85,3.77
pw15.wav,4.37,3.93,4.43,4.47,4.48,4.44,4.31,4.39,4.35


  ✓ Saved per_file_precision.tex


## 5 — Species / Subset Breakdown

TP / FP / FN are pooled per (experiment, subset) and F / P / R are recomputed from the pooled counts, so these are genuine per-subset **micro** scores: every event in the subset carries equal weight. Subsets follow the filename-prefix rule in `parse_logs.per_file_df` (BUK, ME, R4, RD, PW, FILE, OTHER).

In [6]:
if not df_pf.empty:
    # Pooled counts (TP/FP/FN as integers) per (experiment, subset)
    agg = (
        df_pf.groupby(["experiment", "subset"])[["TP", "FP", "FN"]]
        .sum()
        .reset_index()
    )
    # Recompute micro P/R/F from pooled counts, scaled to 0–100
    denom_p = (agg["TP"] + agg["FP"]).clip(lower=1e-9)
    denom_r = (agg["TP"] + agg["FN"]).clip(lower=1e-9)
    denom_f = (agg["TP"] + 0.5 * (agg["FP"] + agg["FN"])).clip(lower=1e-9)
    agg["Precision"] = (agg["TP"] / denom_p) * 100
    agg["Recall"]    = (agg["TP"] / denom_r) * 100
    agg["F-measure"] = (agg["TP"] / denom_f) * 100

    subsets = sorted(agg["subset"].unique())

    def _pivot(metric):
        return (
            agg.pivot_table(index="experiment", columns="subset", values=metric)
            .reindex(index=exp_labels, columns=subsets)
            .round(N_DECIMALS)
        )

    pivot_sub_f = _pivot("F-measure")
    pivot_sub_r = _pivot("Recall")
    pivot_sub_p = _pivot("Precision")

    # Add an arithmetic subset-mean column
    for p in (pivot_sub_f, pivot_sub_r, pivot_sub_p):
        p["Mean (arith.)"] = p[subsets].mean(axis=1).round(N_DECIMALS)

    print("=== Table 4a — F-measure per Subset (Micro pooled; 0–100) ===")
    display(pivot_sub_f)

    save_tex(
        pivot_sub_f,
        "per_subset_fmeasure.tex",
        caption=(
            "Per-species F-measure on the DCASE 2024 Task~5 validation set. "
            "Scores are computed from pooled TP/FP/FN within each subset "
            "(micro within subset) and combined by an unweighted arithmetic mean "
            "across subsets in the last column. The official DCASE score instead "
            "takes a harmonic mean of the same per-subset values. "
            "All values on a 0--100 scale."
        ),
        label="tab:per_subset_fmeasure",
        bold_rows=True,
    )

    print("\n=== Table 4b — Recall per Subset (Micro pooled; 0–100) ===")
    display(pivot_sub_r)

    save_tex(
        pivot_sub_r,
        "per_subset_recall.tex",
        caption=(
            "Per-species recall on the DCASE 2024 Task~5 validation set. "
            "Scores are computed from pooled TP/FN within each subset (micro) "
            "and combined by an unweighted arithmetic mean across subsets in the last column. "
            "All values on a 0--100 scale."
        ),
        label="tab:per_subset_recall",
        bold_rows=True,
    )

    print("\n=== Table 4c — Precision per Subset (Micro pooled; 0–100) ===")
    display(pivot_sub_p)

    save_tex(
        pivot_sub_p,
        "per_subset_precision.tex",
        caption=(
            "Per-species precision on the DCASE 2024 Task~5 validation set. "
            "Scores are computed from pooled TP/FP within each subset (micro) "
            "and combined by an unweighted arithmetic mean across subsets in the last column. "
            "All values on a 0--100 scale."
        ),
        label="tab:per_subset_precision",
        bold_rows=True,
    )

=== Table 4a — F-measure per Subset (Micro pooled; 0–100) ===


subset,HB,ME,PB,PB24,PW,RD,Mean (arith.)
experiment,,,,,,,
Baseline,82.29,25.56,9.78,16.47,8.95,27.57,28.44
CIFE,79.94,41.24,9.23,9.79,8.76,27.65,29.44
Slot Uniform,83.66,19.07,9.04,14.40,9.02,30.36,27.59
Phase 1 (Random),83.70,19.16,8.83,14.89,9.43,29.90,27.65
Phase 2 (Energy),83.70,19.34,8.93,15.14,8.92,28.89,27.49
Phase 3 (Activation),83.98,21.31,8.88,12.57,9.32,30.56,27.77
AFRI (Direct),83.70,23.95,8.42,14.88,9.23,27.63,27.97
AFRI (Inverse),83.77,18.35,8.92,14.78,9.76,28.15,27.29


  ✓ Saved per_subset_fmeasure.tex

=== Table 4b — Recall per Subset (Micro pooled; 0–100) ===


subset,HB,ME,PB,PB24,PW,RD,Mean (arith.)
experiment,,,,,,,
Baseline,79.31,76.92,13.19,9.82,78.57,61.85,53.28
CIFE,78.55,76.92,11.74,5.61,79.52,68.03,53.40
Slot Uniform,80.06,78.85,16.09,9.64,79.84,59.69,54.03
Phase 1 (Random),79.91,78.85,15.65,10.00,80.32,60.58,54.22
Phase 2 (Energy),79.91,78.85,16.09,10.36,79.84,61.03,54.35
Phase 3 (Activation),80.36,75.00,11.74,7.50,79.84,59.61,52.34
AFRI (Direct),79.91,76.92,13.04,9.64,79.37,64.68,53.93
AFRI (Inverse),79.91,76.92,16.52,10.00,80.00,59.91,53.88


  ✓ Saved per_subset_recall.tex

=== Table 4c — Precision per Subset (Micro pooled; 0–100) ===


subset,HB,ME,PB,PB24,PW,RD,Mean (arith.)
experiment,,,,,,,
Baseline,85.50,15.33,7.77,50.91,4.74,17.74,30.33
CIFE,81.38,28.17,7.61,38.10,4.64,17.35,29.54
Slot Uniform,87.60,10.85,6.28,28.42,4.78,20.36,26.38
Phase 1 (Random),87.87,10.90,6.15,29.17,5.01,19.85,26.49
Phase 2 (Energy),87.87,11.02,6.18,28.16,4.72,18.93,26.15
Phase 3 (Activation),87.93,12.42,7.14,38.89,4.95,20.55,28.65
AFRI (Direct),87.87,14.18,6.21,32.53,4.90,17.57,27.21
AFRI (Inverse),88.02,10.42,6.11,28.28,5.20,18.39,26.07


  ✓ Saved per_subset_precision.tex


## 6 — Top-20 Files by Mean False Positives

Which audio files generate the most false positives averaged across all experiments? Also shows mean TP / FN for context.

In [7]:
if not df_pf.empty:
    per_file_counts = (
        df_pf.groupby("filename")[["TP", "FP", "FN"]]
        .mean()
        .rename(columns={"TP": "Mean TP", "FP": "Mean FP", "FN": "Mean FN"})
        .sort_values("Mean FP", ascending=False)
        .head(20)
        .round(N_DECIMALS)
    )
    per_file_counts.insert(0, "Rank", np.arange(1, len(per_file_counts) + 1))

    print("=== Table 5 — Top-20 Files by Mean False Positives (across experiments) ===")
    display(per_file_counts)

    save_tex(
        per_file_counts,
        "top20_fp_files.tex",
        caption=(
            "Top-20 audio files by mean false-positive count, averaged across "
            "all experiments. Mean TP and Mean FN are shown for context. "
            "Counts are per-file event counts."
        ),
        label="tab:top20_fp_files",
    )
else:
    per_file_counts = pd.DataFrame()

=== Table 5 — Top-20 Files by Mean False Positives (across experiments) ===


,Rank,Mean TP,Mean FP,Mean FN
filename,,,,
RD_04.wav,1,84.12,1523.38,141.88
pw4.wav,2,15.62,1394.88,28.38
pw5.wav,3,27.50,1356.75,3.50
pw6.wav,4,39.50,1144.50,1.50
RD_01.wav,5,74.75,1095.62,16.25
pw2.wav,6,21.50,877.25,14.50
pw15.wav,7,33.62,739.75,7.38
pw1.wav,8,25.62,655.25,3.38
pw12.wav,9,32.00,462.50,10.00


  ✓ Saved top20_fp_files.tex


## 7 — TP / FP / FN for a Selected File (across experiments)

Tabular replacement for the stacked-bar drill-down. Defaults to the top-ranked FP file — change `SELECTED_FILE` to inspect any other file.

In [8]:
SELECTED_FILE = per_file_counts.index[0] if not per_file_counts.empty else None

if SELECTED_FILE and not df_pf.empty:
    sub = (
        df_pf[df_pf["filename"] == SELECTED_FILE]
        .set_index("experiment")
        .reindex(exp_labels)
    )
    df_file = sub[["TP", "FP", "FN", "total_n_pos_events",
                   "precision", "recall", "fmeasure"]].copy()
    df_file = df_file.rename(columns={
        "total_n_pos_events": "# events",
        "precision": "Precision",
        "recall":    "Recall",
        "fmeasure":  "F-measure",
    })
    for c in ["Precision", "Recall", "F-measure"]:
        df_file[c] = df_file[c].round(N_DECIMALS)

    print(f"=== Table 6 — TP / FP / FN for {SELECTED_FILE} ===")
    display(df_file)

    _slug = SELECTED_FILE.replace(".", "_").replace(" ", "_")
    save_tex(
        df_file,
        f"selected_file_{_slug}.tex",
        caption=(
            f"Per-experiment detection counts and metrics for "
            f"\texttt{{{SELECTED_FILE.replace('_', r'\_')}}}. "
            f"P, R, F on a 0--100 scale; TP/FP/FN as integer counts."
        ),
        label=f"tab:selected_file_{_slug}",
        bold_rows=True,
    )
else:
    print("No file selected — Table 6 skipped.")

SyntaxError: f-string expression part cannot include a backslash (972174479.py, line 28)

## 8 — Δ F-measure vs Baseline (per file)

Absolute change in F-measure vs. the Baseline column, per file × experiment. Rows sorted by mean absolute Δ (most volatile files first). Positive values ⇒ method beats Baseline on that file.

In [ ]:
if not df_pf.empty and BASELINE_LABEL in exp_labels:
    pivot_fm_raw = (
        df_pf.pivot_table(index="filename", columns="experiment", values="fmeasure")
        .reindex(columns=exp_labels)
    )
    delta = pivot_fm_raw.sub(pivot_fm_raw[BASELINE_LABEL], axis=0).drop(columns=[BASELINE_LABEL])
    delta = delta.loc[delta.abs().mean(axis=1).sort_values(ascending=False).index]
    delta["Mean |Δ|"] = delta.abs().mean(axis=1)
    delta = delta.round(N_DECIMALS)

    print(f"=== Table 7 — Δ F-measure vs {BASELINE_LABEL} (0–100 scale, sorted by |Δ|) ===")
    display(delta)

    save_tex(
        delta,
        "delta_fmeasure_vs_baseline.tex",
        caption=(
            f"Per-file $\\Delta$ F-measure relative to the {BASELINE_LABEL} "
            f"configuration. Rows sorted by mean absolute deviation (most "
            f"volatile files first). Positive values indicate the method beats "
            f"the baseline on that file. All values on a 0--100 scale."
        ),
        label="tab:delta_fmeasure_vs_baseline",
    )
else:
    print(f"Baseline label '{BASELINE_LABEL}' not found — Table 7 skipped.")

## 9 — LaTeX Export

Copy-paste ready `\begin{table}` blocks for a research paper. Exports the per-subset F-measure table (Table 4a) and the per-file F-measure table (Table 1).

In [ ]:
if not df_pf.empty:
    # ── Subset-level F-measure (Table 4a) ──
    latex_subset = pivot_sub_f.to_latex(
        float_format=f"%.{N_DECIMALS}f",
        na_rep="—",
        caption=(
            "Per-species F-measure on the DCASE 2024 Task 5 validation set. "
            "Scores are computed from pooled TP/FP/FN within each subset (micro) and "
            "combined by an unweighted arithmetic mean across subsets in the last column. "
            "All values on a 0--100 scale."
        ),
        label="tab:per_species_fmeasure",
        bold_rows=True,
    )
    print("% ==== Per-subset F-measure ====")
    print(latex_subset)

    # ── Per-file F-measure (Table 1) ──
    latex_file = pivot_fm.to_latex(
        float_format=f"%.{N_DECIMALS}f",
        na_rep="—",
        caption=(
            "Per-file F-measure for every experiment. "
            "F-measure computed as $\\mathrm{TP} / (\\mathrm{TP} + 0.5(\\mathrm{FP} + \\mathrm{FN}))$; "
            "rows sorted ascending by row mean. All values on a 0--100 scale."
        ),
        label="tab:per_file_fmeasure",
    )
    print("\n% ==== Per-file F-measure ====")
    print(latex_file)

% ==== Per-subset F-measure ====
\begin{table}
\centering
\caption{Per-species F-measure on the DCASE 2024 Task 5 validation set. Scores are computed from pooled TP/FP/FN within each subset (micro) and averaged unweighted across subsets in the last column (macro). All values on a 0--100 scale.}
\label{tab:per_species_fmeasure}
\begin{tabular}{lrrrrrr}
\toprule
\textbf{subset} &    HB &    ME &    PB &   PW &    RD &  Mean (arith.) \\
\textbf{experiment              } &       &       &       &      &       &               \\
\midrule
\textbf{Baseline                } & 46.85 & 25.64 & 12.98 & 8.91 & 27.35 &         24.35 \\
\textbf{CIFE                    } & 48.76 & 41.67 &  9.69 & 8.87 & 29.06 &         27.61 \\
\textbf{Phase 1 (Random)        } & 47.39 & 21.73 & 10.73 & 9.23 & 29.84 &         23.78 \\
\textbf{Phase 2 (Energy)        } & 47.39 & 18.98 & 10.81 & 9.15 & 28.80 &         23.03 \\
\textbf{Phase 3 (Activation)    } & 47.39 & 21.43 & 10.31 & 9.32 & 29.96 &         23.68 \\
\